### Case: Score de Crédito dos Clientes
Você foi contratado por um banco para conseguir definir o score de crédito dos clientes. Você precisa analisar todos os clientes do banco e, com base nessa análise, criar um modelo que consiga ler as informações do cliente e dizer automaticamente o score de crédito dele: Ruim, Ok, Bom

### Bibliotecas Utilizadas
- Pandas  
  Utilizada para importar, organizar e manipular os dados da base de clientes.

- Scikit-learn (sklearn)  
  Biblioteca de Machine Learning utilizada para:
  - tratamento dos dados
  - separação entre treino e teste
  - criação dos modelos
  - avaliação dos resultados

### Modelos de Machine Learning Utilizados

#### Random Forest Classifier (Ensemble)
  Modelo baseado em múltiplas árvores de decisão.  

#### K-Nearest Neighbors (KNN)  

  Ele compara novos clientes com outros clientes parecidos já existentes na base de dados.

#### Objetivo dos Modelos

  Os modelos foram treinados utilizando dados históricos dos clientes para aprender padrões financeiros e, posteriormente, prever automaticamente o score de crédito de novos clientes.

In [ ]:
import pandas as pd

# Passo 1: importar a base
tabela = pd.read_csv("clientes.csv")

display(tabela)
display(tabela.info())

In [ ]:
# Identificar as colunas categóricas antes de separar os dados
colunas_categoricas = tabela.select_dtypes(include="object").columns.drop("score_credito").tolist()


In [ ]:
# Separar o alvo, o identificador e as variáveis de entrada
y = tabela["score_credito"]
grupos = tabela["id_cliente"]
x = tabela.drop(columns=["score_credito", "id_cliente"])


In [ ]:
# Manter registros de um mesmo cliente no mesmo conjunto
from sklearn.model_selection import GroupShuffleSplit

separador = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
indices_treino, indices_teste = next(separador.split(x, y, groups=grupos))
x_treino, x_teste = x.iloc[indices_treino], x.iloc[indices_teste]
y_treino, y_teste = y.iloc[indices_treino], y.iloc[indices_teste]


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colunas_numericas = x.select_dtypes(exclude="object").columns.tolist()

def criar_preprocessamento(normalizar=False):
    etapas_numericas = [SimpleImputer(strategy="median")]
    if normalizar:
        etapas_numericas.append(StandardScaler())
    return ColumnTransformer([
        ("numericas", make_pipeline(*etapas_numericas), colunas_numericas),
        ("categoricas", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                      OneHotEncoder(handle_unknown="ignore")), colunas_categoricas),
    ])

modelo_arvoredecisao = make_pipeline(criar_preprocessamento(), RandomForestClassifier(random_state=1, n_jobs=-1))
modelo_knn = make_pipeline(criar_preprocessamento(normalizar=True), KNeighborsClassifier())
modelo_arvoredecisao.fit(x_treino, y_treino)
modelo_knn.fit(x_treino, y_treino)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

resultados = {}
for nome, modelo in [("Random Forest", modelo_arvoredecisao), ("KNN", modelo_knn)]:
    previsao = modelo.predict(x_teste)
    relatorio = classification_report(y_teste, previsao, output_dict=True, zero_division=0)
    resultados[nome] = (modelo, previsao, relatorio)

resumo = pd.DataFrame({
    nome: {
        "Acurácia": accuracy_score(y_teste, previsao),
        "Precisão (macro)": relatorio["macro avg"]["precision"],
        "Recall (macro)": relatorio["macro avg"]["recall"],
        "F1-score (macro)": relatorio["macro avg"]["f1-score"],
    }
    for nome, (_, previsao, relatorio) in resultados.items()
}).T
print("COMPARAÇÃO DOS MODELOS")
display(resumo.map(lambda valor: f"{valor:.2%}"))

for nome, (modelo, previsao, _) in resultados.items():
    print(f"\nDETALHAMENTO — {nome}")
    print(classification_report(y_teste, previsao, zero_division=0))
    print("Matriz de confusão:")
    classes = modelo.classes_
    display(pd.DataFrame(confusion_matrix(y_teste, previsao, labels=classes),
                         index=classes, columns=classes).rename_axis("Real").rename_axis("Previsto", axis=1))


In [ ]:
# Prever os scores de novos clientes com o mesmo pré-processamento do treino
novos_clientes = pd.read_csv("novos_clientes.csv")
previsao = modelo_arvoredecisao.predict(novos_clientes)
display(novos_clientes.assign(score_previsto=previsao))
